# DQN Pong — Kelvin's k01–k10

Reruns my ten assigned experiments (k01–k10) on **ALE/Pong-v5** using **Paulette's train.py verbatim**, so my results are directly comparable with the group's results.csv (her baseline scored −14.3, best +9.0 at 500k steps).

Reliability: the working directory IS the Drive folder, so results.csv, every model, TensorBoard logs and the run log are written straight to Drive as they happen. Each experiment gets a DONE flag; if the session ever dies, reopen → Run all → completed experiments are skipped and it resumes.


## 1 · Install (once → then Restart session)

In [1]:
!pip install -q "stable-baselines3[extra]" "gymnasium[atari]" "ale-py" "autorom[accept-rom-license]"
print("Installed. Runtime -> Restart session, then Runtime -> Run all.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 434.7/434.7 kB 11.7 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 19.3 MB/s eta 0:00:00
Installed. Runtime -> Restart session, then Runtime -> Run all.


## 2 · Checks + Drive workspace (everything runs INSIDE Drive)

In [2]:
import torch
assert torch.cuda.is_available(), "STOP: Runtime -> Change runtime type -> GPU (T4 is enough; A100 wastes units here)"
print("GPU:", torch.cuda.get_device_name(0))

import ale_py, gymnasium as gym
gym.register_envs(ale_py)
gym.make("ALE/Pong-v5")
print("ALE OK (Pong)")

from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = "/content/drive/MyDrive/dqn_pong_kelvin"
os.makedirs(f"{DRIVE_DIR}/flags", exist_ok=True)
os.chdir(DRIVE_DIR)   # relative paths in train.py (results.csv, models, tb_logs) now live on Drive
print("Working directory:", os.getcwd())

GPU: Tesla T4
ALE OK (Pong)
Mounted at /content/drive
Working directory: /content/drive/MyDrive/dqn_pong_kelvin


## 3 · Paulette's train.py — VERBATIM (do not edit: comparability depends on it)

In [3]:
%%writefile train.py
"""
train.py - Train a DQN agent on Atari Pong using Stable Baselines3.

Can be used two ways:
  1. From the command line:
     python train.py --name exp02_high_lr --lr 0.001 --timesteps 150000
  2. Imported in a notebook:
     from train import train_dqn
     train_dqn(name="exp02_high_lr", lr=1e-3)

Each run saves the model as dqn_model_<name>.zip, logs to TensorBoard,
and appends a summary row (final evaluation reward, etc.) to results.csv.
"""

import argparse
import csv
import os
import time

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.vec_env import VecFrameStack

import ale_py
import gymnasium as gym
gym.register_envs(ale_py)   # registers the Atari environments with Gymnasium

ENV_ID = "ALE/Pong-v5"      # modern naming; old PongNoFrameskip-v4 was removed
RESULTS_FILE = "results.csv"


def make_env(seed=42):
    """Standard Atari setup: preprocessing + 4-frame stacking so the
    agent can perceive motion (one frame can't show ball direction)."""
    env = make_atari_env(
        ENV_ID, n_envs=1, seed=seed,
        env_kwargs={"frameskip": 1, "repeat_action_probability": 0.0},
    )
    env = VecFrameStack(env, n_stack=4)
    return env


def train_dqn(
    name,
    lr=1e-4,
    gamma=0.99,
    batch_size=32,
    eps_start=1.0,
    eps_end=0.05,
    exploration_fraction=0.1,   # SB3's "epsilon decay": fraction of training
                                # over which epsilon decays linearly.
                                # Smaller = faster decay.
    policy="CnnPolicy",
    timesteps=150_000,
    seed=42,
):
    """Train one DQN agent with the given hyperparameters, save the model,
    evaluate it greedily, and log a summary row to results.csv."""
    env = make_env(seed)

    model = DQN(
        policy=policy,
        env=env,
        learning_rate=lr,
        gamma=gamma,
        batch_size=batch_size,
        exploration_initial_eps=eps_start,
        exploration_final_eps=eps_end,
        exploration_fraction=exploration_fraction,
        buffer_size=50_000,        # replay memory (kept modest for Kaggle RAM)
        learning_starts=10_000,    # random play before learning begins
        train_freq=4,
        target_update_interval=1_000,
        verbose=1,
        tensorboard_log=f"./tb_logs/{name}",
        seed=seed,
    )

    print(f"\n{'='*60}")
    print(f"=== {name}: policy={policy}, lr={lr}, gamma={gamma}, "
          f"batch={batch_size}, eps={eps_start}->{eps_end} "
          f"over {exploration_fraction*100:.0f}% of training ===")
    print(f"{'='*60}\n")

    start = time.time()
    model.learn(total_timesteps=timesteps, log_interval=25)
    minutes = (time.time() - start) / 60

    model.save(f"dqn_model_{name}")

    # Greedy evaluation (deterministic=True = GreedyQPolicy): how good is
    # the agent when it stops exploring and just plays its best?
    eval_env = make_env(seed=123)
    mean_reward, std_reward = evaluate_policy(
        model, eval_env, n_eval_episodes=3, deterministic=True
    )
    eval_env.close()
    env.close()

    print(f"\n>>> {name} done in {minutes:.1f} min | "
          f"greedy eval reward: {mean_reward:.1f} +/- {std_reward:.1f}\n")

    # Append summary row to results.csv -> this becomes your table
    write_header = not os.path.exists(RESULTS_FILE)
    with open(RESULTS_FILE, "a", newline="") as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow([
                "experiment", "policy", "lr", "gamma", "batch_size",
                "eps_start", "eps_end", "exploration_fraction",
                "timesteps", "train_minutes", "eval_reward_mean",
                "eval_reward_std",
            ])
        writer.writerow([
            name, policy, lr, gamma, batch_size, eps_start, eps_end,
            exploration_fraction, timesteps, round(minutes, 1),
            round(mean_reward, 2), round(std_reward, 2),
        ])

    return mean_reward


def parse_args():
    p = argparse.ArgumentParser(description="Train a DQN agent on Atari Pong")
    p.add_argument("--name", required=True, help="experiment name, e.g. exp01_baseline")
    p.add_argument("--lr", type=float, default=1e-4)
    p.add_argument("--gamma", type=float, default=0.99)
    p.add_argument("--batch", type=int, default=32)
    p.add_argument("--eps_start", type=float, default=1.0)
    p.add_argument("--eps_end", type=float, default=0.05)
    p.add_argument("--exploration_fraction", type=float, default=0.1)
    p.add_argument("--policy", default="CnnPolicy", choices=["CnnPolicy", "MlpPolicy"])
    p.add_argument("--timesteps", type=int, default=150_000)
    return p.parse_args()


if __name__ == "__main__":
    args = parse_args()
    train_dqn(
        name=args.name,
        lr=args.lr,
        gamma=args.gamma,
        batch_size=args.batch,
        eps_start=args.eps_start,
        eps_end=args.eps_end,
        exploration_fraction=args.exploration_fraction,
        policy=args.policy,
        timesteps=args.timesteps,
    )

Overwriting train.py


In [4]:
import os, sys, importlib

# 1. Remove the stale Breakout-era script that's shadowing the real one
!rm -f /content/train.py

# 2. Make sure we're inside the Drive workspace
DRIVE_DIR = "/content/drive/MyDrive/dqn_pong_kelvin"
os.chdir(DRIVE_DIR)
print("cwd:", os.getcwd())

# 3. If Drive's train.py is missing, cell 3 (%%writefile) needs rerunning first
assert os.path.exists("train.py"), "train.py not here -> rerun Cell 3, then this cell"

# 4. Purge any cached import of the wrong module, then import fresh
if "train" in sys.modules:
    del sys.modules["train"]
importlib.invalidate_caches()

import train
from train import train_dqn
print("Imported from:", train.__file__)   # MUST end with /dqn_pong_kelvin/train.py

cwd: /content/drive/MyDrive/dqn_pong_kelvin


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


Imported from: /content/drive/MyDrive/dqn_pong_kelvin/train.py


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 4 · Resilient runner
Wraps her `train_dqn` with resume flags, error isolation, logging and memory cleanup. Her function itself is untouched.

In [5]:
import os, gc, time, traceback
import torch
import importlib
import train
importlib.reload(train)
from train import train_dqn

def log_line(msg):
    line = f"[{time.strftime('%Y-%m-%d %H:%M:%S')}] {msg}"
    print(line, flush=True)
    with open("run_log.txt", "a") as f:
        f.write(line + "\n")

def run_safe(config, timesteps):
    name = config["name"]
    flag = f"flags/{name}.DONE"
    if os.path.exists(flag):
        log_line(f"SKIP {name} (already completed)")
        return True
    try:
        log_line(f"START {name} @ {timesteps} steps: {config}")
        train_dqn(timesteps=timesteps, **config)
        open(flag, "w").write("done")
        log_line(f"DONE {name}")
        return True
    except Exception as e:
        log_line(f"FAILED {name}: {e}")
        with open("errors.txt", "a") as f:
            f.write(f"\n=== {name} ===\n{traceback.format_exc()}\n")
        return False
    finally:
        gc.collect()
        torch.cuda.empty_cache()

log_line("Runner ready")

[2026-07-18 16:34:50] Runner ready


## 5 · My ten experiments (from the team execution plan)
`TIMESTEPS = 500_000` matches Paulette's and David's runs. Fallback if badly behind schedule tomorrow morning: change to 300_000 for the REMAINING experiments only, and note it in the write-up.

In [6]:
TIMESTEPS = 500_000

EXPERIMENTS = [
    dict(name="k01_lr_5e4", lr=5e-4),
    dict(name="k02_lr_5e5", lr=5e-5),
    dict(name="k03_gamma_095", gamma=0.95),
    dict(name="k04_gamma_098", gamma=0.98),
    dict(name="k05_batch16", batch_size=16),
    dict(name="k06_batch256", batch_size=256),
    dict(name="k07_epsend_001", eps_end=0.01),
    dict(name="k08_epsend_015", eps_end=0.15),
    dict(name="k09_decay_5pct", exploration_fraction=0.05),
    dict(name="k10_decay_50pct", exploration_fraction=0.5),
]

status = {c["name"]: run_safe(c, TIMESTEPS) for c in EXPERIMENTS}

failed = [n for n, ok in status.items() if not ok]
if failed:
    log_line(f"Finished with failures: {failed}. Rerun THIS cell to retry only those (see errors.txt).")
else:
    log_line("All 10 experiments completed.")

[2026-07-18 16:34:50] SKIP k01_lr_5e4 (already completed)
[2026-07-18 16:34:50] SKIP k02_lr_5e5 (already completed)
[2026-07-18 16:34:50] SKIP k03_gamma_095 (already completed)
[2026-07-18 16:34:50] SKIP k04_gamma_098 (already completed)
[2026-07-18 16:34:50] SKIP k05_batch16 (already completed)
[2026-07-18 16:34:50] SKIP k06_batch256 (already completed)
[2026-07-18 16:34:50] SKIP k07_epsend_001 (already completed)
[2026-07-18 16:34:50] START k08_epsend_015 @ 500000 steps: {'name': 'k08_epsend_015', 'eps_end': 0.15}
Using cuda device
Wrapping the env in a VecTransposeImage.

=== k08_epsend_015: policy=CnnPolicy, lr=0.0001, gamma=0.99, batch=32, eps=1.0->0.15 over 10% of training ===

Logging to ./tb_logs/k08_epsend_015/DQN_2


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.71e+03 |
|    ep_rew_mean      | -20.2    |
|    exploration_rate | 0.608    |
| time/               |          |
|    episodes         | 25       |
|    fps              | 417      |
|    time_elapsed     | 55       |
|    total_timesteps  | 23030    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 2.43e-05 |
|    n_updates        | 3257     |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 3.61e+03 |
|    ep_rew_mean      | -20.4    |
|    exploration_rate | 0.237    |
| time/               |          |
|    episodes         | 50       |
|    fps              | 354      |
|    time_elapsed     | 126      |
|    total_timesteps  | 44901    |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.003    |
|    n_updates      

## 6 · Deliverables: results_kelvin.csv + ranking

In [7]:
import pandas as pd, shutil

df = pd.read_csv("results.csv")
mine = df[df["experiment"].str.startswith("k")].copy()
mine.to_csv("results_kelvin.csv", index=False)
print(f"results_kelvin.csv written with {len(mine)} rows (send this to Paulette)\n")

ranked = mine.sort_values("eval_reward_mean", ascending=False)
print(ranked[["experiment","lr","gamma","batch_size","eps_end",
              "exploration_fraction","eval_reward_mean","eval_reward_std","train_minutes"]].to_string(index=False))
print(f"\nReference points from Paulette's runs: baseline -14.3, best +9.0 (batch 128)")

results_kelvin.csv written with 10 rows (send this to Paulette)

     experiment      lr  gamma  batch_size  eps_end  exploration_fraction  eval_reward_mean  eval_reward_std  train_minutes
   k06_batch256 0.00010   0.99         256     0.05                  0.10             19.67             1.89           56.2
k10_decay_50pct 0.00010   0.99          32     0.05                  0.50             -6.33             0.47           27.7
     k02_lr_5e5 0.00005   0.99          32     0.05                  0.10             -7.67             0.94           31.1
  k03_gamma_095 0.00010   0.95          32     0.05                  0.10             -9.33             7.54           32.3
 k09_decay_5pct 0.00010   0.99          32     0.05                  0.05             -9.67             0.94           30.4
 k08_epsend_015 0.00010   0.99          32     0.15                  0.10            -15.00             7.07           29.5
  k04_gamma_098 0.00010   0.98          32     0.05                

/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


## 7 · Noted-behavior skeletons
Prints a starter paragraph per experiment with my real numbers filled in, compared against the group baseline. I edit these into my own words for the README and the presentation, checking each against the actual result rather than the expectation.

In [8]:
BASELINE = -14.3   # Paulette's exp01_baseline at 500k

reasons = {
 "k01_lr_5e4": "a 5x higher learning rate than baseline, larger Q updates",
 "k02_lr_5e5": "a 2x lower learning rate, smaller and slower Q updates",
 "k03_gamma_095": "a shorter reward horizon (gamma 0.95)",
 "k04_gamma_098": "a slightly shortened reward horizon (gamma 0.98)",
 "k05_batch16": "small noisy gradient batches of 16",
 "k06_batch256": "very large smooth gradient batches of 256",
 "k07_epsend_001": "an epsilon floor of 0.01, almost pure exploitation late in training",
 "k08_epsend_015": "an epsilon floor of 0.15, permanent 15 percent random actions",
 "k09_decay_5pct": "epsilon decayed within the first 5 percent of training",
 "k10_decay_50pct": "epsilon decayed over half of training",
}

for _, r in mine.iterrows():
    delta = r["eval_reward_mean"] - BASELINE
    verdict = "better than" if delta > 1 else ("worse than" if delta < -1 else "about the same as")
    print(f"--- {r['experiment']} ---")
    print(f"This run used {reasons.get(r['experiment'],'')}. It scored "
          f"{r['eval_reward_mean']:.1f} +/- {r['eval_reward_std']:.1f}, which is {verdict} "
          f"the group baseline of {BASELINE} ({'+' if delta>=0 else ''}{delta:.1f}). "
          f"[I explain WHY here in 1-2 sentences based on what I saw in the training log.]\n")

--- k01_lr_5e4 ---
This run used a 5x higher learning rate than baseline, larger Q updates. It scored -21.0 +/- 0.0, which is worse than the group baseline of -14.3 (-6.7). [I explain WHY here in 1-2 sentences based on what I saw in the training log.]

--- k02_lr_5e5 ---
This run used a 2x lower learning rate, smaller and slower Q updates. It scored -7.7 +/- 0.9, which is better than the group baseline of -14.3 (+6.6). [I explain WHY here in 1-2 sentences based on what I saw in the training log.]

--- k03_gamma_095 ---
This run used a shorter reward horizon (gamma 0.95). It scored -9.3 +/- 7.5, which is better than the group baseline of -14.3 (+5.0). [I explain WHY here in 1-2 sentences based on what I saw in the training log.]

--- k04_gamma_098 ---
This run used a slightly shortened reward horizon (gamma 0.98). It scored -16.0 +/- 2.8, which is worse than the group baseline of -14.3 (-1.7). [I explain WHY here in 1-2 sentences based on what I saw in the training log.]

--- k05_batch1

## 8 · Done
On Drive in `dqn_pong_kelvin/`: `results_kelvin.csv` (→ Paulette before 8pm), `results.csv`, one `dqn_model_k*.zip` per experiment, `tb_logs/`, `run_log.txt`, `errors.txt` if anything failed, and DONE flags. My old Breakout folder stays untouched as backup and stays out of the repo.

In [9]:
!ls -lh dqn_model_k*.zip results_kelvin.csv 2>/dev/null
!tail -15 run_log.txt

-rw------- 1 root root 26M Jul 18 10:22 dqn_model_k01_lr_5e4.zip
-rw------- 1 root root 26M Jul 18 10:53 dqn_model_k02_lr_5e5.zip
-rw------- 1 root root 26M Jul 18 13:15 dqn_model_k03_gamma_095.zip
-rw------- 1 root root 26M Jul 18 13:47 dqn_model_k04_gamma_098.zip
-rw------- 1 root root 26M Jul 18 14:18 dqn_model_k05_batch16.zip
-rw------- 1 root root 26M Jul 18 15:14 dqn_model_k06_batch256.zip
-rw------- 1 root root 26M Jul 18 15:47 dqn_model_k07_epsend_001.zip
-rw------- 1 root root 26M Jul 18 17:04 dqn_model_k08_epsend_015.zip
-rw------- 1 root root 26M Jul 18 17:35 dqn_model_k09_decay_5pct.zip
-rw------- 1 root root 26M Jul 18 18:03 dqn_model_k10_decay_50pct.zip
-rw------- 1 root root 880 Jul 18 18:03 results_kelvin.csv
[2026-07-18 16:34:50] Runner ready
[2026-07-18 16:34:50] SKIP k01_lr_5e4 (already completed)
[2026-07-18 16:34:50] SKIP k02_lr_5e5 (already completed)
[2026-07-18 16:34:50] SKIP k03_gamma_095 (already completed)
[2026-07-18 16:34:50] SKIP k04_gamma_098 (already com